# 第 8 章 流式、异步与并发（Streaming & Async）

**学习目标**
- 掌握 `stream`：让回答像 ChatGPT 一样"打字机"式输出
- 理解 `AIMessageChunk` 及其累加方式
- 整条 LCEL 链的流式（不是只有模型能流式）
- 会用 `asyncio` 并发调用多个 LLM 任务
- 会用 `astream_events` 观察链内部的事件流

> 流式决定**体验**（用户 2 秒内看到第一个字），异步决定**吞吐**（100 个请求并行处理）。两者都是生产应用必备。

In [ ]:
# 环境准备：加载 .env 并初始化模型（本系列所有章节通用）
import os
from dotenv import load_dotenv

load_dotenv()  # 读取项目根目录下的 .env 文件

from langchain_openai import ChatOpenAI

# 智谱 GLM 提供 OpenAI 兼容接口，直接用 ChatOpenAI 指向智谱的 base_url 即可
llm = ChatOpenAI(
    model="glm-4.5-air",  # 可换成你账号可用的其他模型（改模型名即可，无需改其他代码）
    temperature=0.6,
    api_key=os.getenv("API_KEY_ZHIPU"),
    base_url=os.getenv("API_URL_ZHIPU"),  # 形如 https://open.bigmodel.cn/api/paas/v4/
)

print("模型就绪:", llm.model_name)
# 注意：本章会发起若干次真实 API 调用，会产生少量费用；练习请使用短文本。

## 1. stream：模型级流式

`llm.stream()` 返回一个生成器，逐段产出 `AIMessageChunk`。每个 chunk 的 `content` 只是增量片段，需要自己拼接。

In [ ]:
from langchain_core.messages import AIMessageChunk

print("打字机效果:")
final_chunk = None
for chunk in llm.stream("用三句话介绍什么是向量数据库"):
    print(chunk.content, end="", flush=True)
    final_chunk = chunk if final_chunk is None else final_chunk + chunk

print("\n\n--- 流结束后 ---")
print("累加后的完整内容类型:", type(final_chunk))
print("完整内容:", final_chunk.content)

> **chunk 对象可以相加**：`chunk1 + chunk2` 会把内容、tool_calls 等字段按规则合并，最后得到一个完整的 `AIMessageChunk`。需要"完整回复"做后续处理（如解析 JSON）时就是这么攒出来的。

## 2. 整条链的流式

LCEL 的杀手锏：只要链中每个环节都支持流式，`chain.stream()` 就是端到端流式——解析器会逐块透传，你拿到的就是增量文本，无需关心链里有多少环节。

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

chain = (
    ChatPromptTemplate.from_messages([
        ("system", "你是资深技术布道师，回答分 3 个要点，每点不超过 30 字"),
        ("human", "为什么{topic}值得学习？"),
    ])
    | llm
    | StrOutputParser()
)

for piece in chain.stream({"topic": "LangChain"}):
    print(piece, end="", flush=True)

## 3. batch 与 abatch：并发批处理

第 4 章已经用过 `batch`：传入输入列表，LangChain 自动并发执行（默认并发上限可配）。异步版 `abatch` 用法相同，适合已有事件循环的场景（如 FastAPI 服务里）。

In [ ]:
import time

questions = [
    {"topic": "向量数据库"}, {"topic": "微服务"}, {"topic": "函数式编程"},
]

t0 = time.time()
results = chain.batch(questions)
t1 = time.time()
print(f"batch 并发 3 个请求耗时 {t1 - t0:.1f}s（顺序执行约需 3 倍单次耗时）")
for r in results:
    print("•", r[:40], "...")

## 4. 异步基础：ainvoke / astream

Notebook（IPython）顶层直接支持 `await`。所有 Runnable 都有 `a` 前缀的异步版本：`ainvoke / astream / abatch`。

In [ ]:
import asyncio

# 异步流式
async def demo_astream():
    async for piece in chain.astream({"topic": "RAG"}):
        print(piece, end="", flush=True)

await demo_astream()

In [ ]:
# 并发调度多个异步任务：asyncio.gather
import time

async def answer(topic: str) -> str:
    return await chain.ainvoke({"topic": topic})

async def demo_concurrent():
    t0 = time.time()
    tasks = [answer(t) for t in ["并发", "缓存", "异步 IO", "消息队列"]]
    results = await asyncio.gather(*tasks)     # 4 个请求同时发出
    print(f"并发 4 个请求总耗时 {time.time() - t0:.1f}s\n")
    for r in results:
        print("•", r[:40], "...")

await demo_concurrent()

> **asyncio.gather 是异步并发的核心**：把多个协程打包同时执行，全部完成后按顺序返回结果列表。配合 `asyncio.Semaphore` 可限制并发数，防止触发 API 限流。

## 5. astream_events：观察链的"事件流"

比 `astream` 更底层的观测手段：链执行过程中每个环节（prompt、chat model、parser）都会发出事件。做调试、做 Web 应用的事件推送（SSE）都靠它。

In [ ]:
async def demo_events():
    async for event in chain.astream_events({"topic": "Agent"}, version="v2"):
        if event["event"] == "on_chat_model_stream":        # 模型吐字的每个 chunk
            print(event["data"]["chunk"].content, end="", flush=True)
        elif event["event"] == "on_chain_start":
            print(f"▶ 链开始: {event['name']}")
        elif event["event"] == "on_chain_end":
            print(f"\n✔ 链结束: {event['name']}")

await demo_events()

常用事件：
| 事件 | 含义 |
|---|---|
| `on_chain_start` / `on_chain_end` | 某个 Runnable（含整条链）开始/结束 |
| `on_chat_model_start` | 模型开始 |
| `on_chat_model_stream` | 模型流式吐出一个 chunk |
| `on_chat_model_end` | 模型完成（data 里有完整输出） |

---

## 练习

**练习 1：终端流式聊天机器人**
用 `MessagesPlaceholder` + 手动维护历史列表 + `stream`，实现一个多轮流式聊天：每轮打印"你: "、"AI: "（流式输出），把对话追加进历史。跑 3 轮验证它记得之前说的话。

**练习 2：并发摘要器**
给定 5 个长句列表，用 `asyncio.gather` + `abatch`/`ainvoke` 并发生成一句话摘要，比较 `batch` 与手动 `gather` 两种写法的耗时。

**练习 3：事件打点（选做）**
用 `astream_events` 统计一条链执行过程中 `on_chat_model_stream` 事件总数和总输出字符数。

> 提示：练习 1 的关键是**每次对话后把 HumanMessage 和完整 AIMessage 都 append 进历史**；流式时先攒 chunk，打印完再把 `final_chunk` 存入历史。

---

## 参考答案（先自己写再看！）

In [ ]:
# 练习 1 参考答案：终端流式聊天机器人
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.output_parsers import StrOutputParser

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是轻松幽默的中文聊天伙伴，回答不超过 40 字"),
    MessagesPlaceholder("history"),
    ("human", "{input}"),
])
chat_chain = chat_prompt | llm | StrOutputParser()

history = []

def chat(user_input: str):
    print("你:", user_input)
    print("AI: ", end="")
    full = ""
    for piece in chat_chain.stream({"history": history, "input": user_input}):
        print(piece, end="", flush=True)
        full += piece                       # 边流式打印边攒完整回复
    print()
    # 关键：把"用户输入"和"攒出来的完整回复"一起写回历史，下一轮才有上下文
    history.extend([HumanMessage(content=user_input), AIMessage(content=full)])

chat("你好，我昨天去爬山了")
chat("猜猜我爬的是哪座山？提示：在广州")
chat("对！山上的空气真好，你说为什么山上空气更清新？")
print(f"\n历史消息条数: {len(history)}")

In [ ]:
# 练习 2 参考答案：并发摘要器
import asyncio, time
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

texts = [
    "大语言模型通过在海量文本上进行预训练，学会了语言的统计规律，再经过指令微调和对齐训练，能够理解并执行人类的自然语言指令。",
    "向量检索的原理是把文本转换成高维向量，通过计算向量之间的余弦相似度来衡量语义相似程度，从而实现比关键词匹配更智能的搜索。",
    "微服务架构将单体应用拆分为多个独立部署的小服务，每个服务专注单一业务能力，通过 API 网关和消息队列协同工作。",
    "缓存通过把热点数据放在更快的存储层来减少重复计算和数据库访问，是提升系统吞吐量的第一手段，但要小心缓存与数据库的一致性问题。",
    "异步编程允许程序在等待 IO 时去做其他事情，事件循环负责调度这些等待中的任务，从而用少量线程支撑高并发。",
]

summarize = (ChatPromptTemplate.from_messages([
    ("system", "把输入压缩成不超过 15 字的摘要"),
    ("human", "{text}"),
]) | llm | StrOutputParser())

# 写法 A：batch（同步 API，内部自动并发）
t0 = time.time()
res_a = summarize.batch(texts)
print(f"batch: {time.time()-t0:.1f}s -> {res_a[:2]}")

# 写法 B：手动 gather（异步 API）
async def run_b():
    t0 = time.time()
    res_b = await asyncio.gather(*[summarize.ainvoke({"text": t}) for t in texts])
    print(f"gather: {time.time()-t0:.1f}s -> {res_b[:2]}")

await run_b()

In [ ]:
# 练习 3 参考答案：事件统计
async def count_events():
    n_chunks, n_chars = 0, 0
    async for event in chain.astream_events({"topic": "微服务"}, version="v2"):
        if event["event"] == "on_chat_model_stream":
            c = event["data"]["chunk"].content
            n_chunks += 1
            n_chars += len(c)
    print(f"stream chunk 数: {n_chunks}，总输出字符数: {n_chars}")

await count_events()